In [ ]:
# Notebook imports
# Generated from standard/third-party imports used throughout this notebook.
import json
import matplotlib.pyplot as plt
import pandas as pd
import sys
import xarray as xr
from pathlib import Path


# WAC Scratch

Exploratory cells for WAC datasets: instance label comparisons and instance segmentation datamodule sanity checks.


# WAC

## iseg label comparison

In [ ]:

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "scratch.ipynb").exists():
    NOTEBOOK_DIR = (Path.cwd() / "notebooks" / "full_model").resolve()
LFM_ROOT = NOTEBOOK_DIR.parents[1]

if str(LFM_ROOT) not in sys.path:
    sys.path.insert(0, str(LFM_ROOT))

from lfm.all_models.all_tasks.utils import create_timestamped_output_dir, plot_instance_label_comparison

KAGUYA_ISEG_ROOT = Path("/panfs/ccds02/nobackup/projects/lfm/model_inputs/300_300_inputs/kaguya_static_all_wac/inst_seg")
NEW_ISEG_ROOT = Path("/panfs/ccds02/nobackup/projects/lfm/model_inputs/300_300_inputs/full_model_inst_seg_v2")
ISEG_LABEL_COMPARISON_OUTPUT_DIR = create_timestamped_output_dir(NOTEBOOK_DIR / "outputs" / "iseg_label_comparison")

plot_instance_label_comparison(
    kaguya_root=KAGUYA_ISEG_ROOT,
    split_data_root=NEW_ISEG_ROOT,
    output_dir=ISEG_LABEL_COMPARISON_OUTPUT_DIR,
    n_samples=8,
    filename="iseg_label_comparison.png",
    display_method="minmax",
    dpi=200,
)

## iseg sanity test

In [ ]:

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "scratch.ipynb").exists():
    NOTEBOOK_DIR = (Path.cwd() / "notebooks" / "full_model").resolve()
LFM_ROOT = NOTEBOOK_DIR.parents[1]
GRAHA_ROOT = LFM_ROOT / "lfm" / "full_model" / "graha-lunar-fm"

PRETRAIN_DIR = Path(
    "/explore/nobackup/projects/lfm/gabby/Lunar-FM/experiments/"
    "lunarfm_base_dual_full_nas_no_nans_256_256_lr1e-4_wd0.05"
).resolve()
BACKBONE_WEIGHTS = PRETRAIN_DIR / "checkpoints/checkpoint_weights_final.pt"
BACKBONE_CFG = PRETRAIN_DIR / "full_config.yaml"
MODALITY_INFO = PRETRAIN_DIR / "modality_info.yaml"
NORMALIZED_WAC_DATA_RANGE = [-1.0, 1.0]

for import_path in [GRAHA_ROOT, LFM_ROOT]:
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from lfm.full_model.inst_seg.instance_mask_datamodule import (
    LunarInstanceMaskSegmentationDatamodule,
    LunarObjectDetectionInstanceMaskDatamodule,
)
from lfm.all_models.all_tasks.utils import create_timestamped_output_dir, plot_instance_batch_sanity
from lfm.all_models.all_tasks.utils.utils import ensure_data_symlink

print("Notebook directory:", NOTEBOOK_DIR)
print("LFM root:", LFM_ROOT)
print("Graha/Lunar-FM code root:", GRAHA_ROOT)
print("Backbone weights:", BACKBONE_WEIGHTS)

In [ ]:
ISEG_DATA_ROOT = Path("/panfs/ccds02/nobackup/projects/lfm/model_inputs/300_300_inputs/full_model_inst_seg_v2")  # expects train/val/test/{chips,labels}
MASK_SHIFT = (0, 0)  # (x_pixels, y_pixels): positive moves labels right/down
ISEG_OUTPUT_DIR = create_timestamped_output_dir(NOTEBOOK_DIR / "outputs" / "instance_sanity")

iseg_datamodule = LunarInstanceMaskSegmentationDatamodule(
    data_root=ISEG_DATA_ROOT,
    crop_size=256,
    batch_size=5,
    num_workers=0,
    mask_shift=MASK_SHIFT,
)

plot_instance_batch_sanity(
    iseg_datamodule,
    output_dir=ISEG_OUTPUT_DIR,
    split="train",
    n_samples=5,
)

### ObjectDetectionTask target sanity test

This checks the true instance target format expected by TerraTorch `ObjectDetectionTask`: `image`, `boxes`, `labels`, and `masks`.

In [ ]:
OD_ISEG_DATA_ROOT = ISEG_DATA_ROOT

od_iseg_datamodule = LunarObjectDetectionInstanceMaskDatamodule(
    data_root=OD_ISEG_DATA_ROOT,
    crop_size=256,
    batch_size=4,
    num_workers=0,
    mask_shift=MASK_SHIFT,
    target_box_format="xyxy",  # pixel xyxy for mask-rcnn
)
od_iseg_datamodule.setup("fit")
od_batch = next(iter(od_iseg_datamodule.train_dataloader()))

print("batch keys:", od_batch.keys())
print("image:", tuple(od_batch["image"].shape), od_batch["image"].dtype)
print("boxes per image:", [tuple(x.shape) for x in od_batch["boxes"]])
print("labels per image:", [tuple(x.shape) for x in od_batch["labels"]])
print("masks per image:", [tuple(x.shape) for x in od_batch["masks"]])
print("first filename:", od_batch["filename"][0])

for i, (boxes, labels, masks) in enumerate(zip(od_batch["boxes"], od_batch["labels"], od_batch["masks"])):
    assert boxes.ndim == 2 and boxes.shape[-1] == 4
    assert labels.ndim == 1
    assert masks.ndim == 3 and masks.shape[-2:] == od_batch["image"].shape[-2:]
    assert boxes.shape[0] == labels.shape[0] == masks.shape[0]
    if boxes.numel():
        assert bool((boxes[:, 2] > boxes[:, 0]).all())
        assert bool((boxes[:, 3] > boxes[:, 1]).all())
print("ObjectDetectionTask target sanity check passed.")